In [ ]:
# ============================================================
# CELL 1 — IMPORTS AND REPRODUCIBILITY SEED
# ============================================================
# We import all the libraries we need upfront:
#   - os, time, random: standard Python utilities
#     os: for checking file paths (cache logic)
#     time: for measuring epoch duration
#     random: for seeding Python's built-in RNG
#   - numpy / pandas: numerical arrays and DataFrame operations
#   - torch: the PyTorch deep learning framework
#   - Dataset, DataLoader: PyTorch utilities for wrapping data
#     into batches and feeding them to the model
#   - AutoTokenizer: automatically loads the correct tokenizer
#     for any HuggingFace model by name
#   - AutoModelForSequenceClassification: loads a pretrained
#     transformer and attaches a new linear classification head
#     on top of the [CLS] token representation
#   - get_linear_schedule_with_warmup: creates a learning rate
#     scheduler that linearly increases LR from 0 to target
#     over warmup steps, then linearly decays back to 0
#   - sklearn metrics: accuracy, F1, precision, recall
#   - matplotlib: for plotting training curves
#
# SEED = 42 ensures full reproducibility:
#   - same train/val split every run
#   - same weight initialization order
#   - same batch shuffling order


import os, time, random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, classification_report
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)        
np.random.seed(SEED)     
torch.manual_seed(SEED)  
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)  

print("Imports done. CUDA:", torch.cuda.is_available())

In [ ]:
# ============================================================
# CELL 2 — DATA LOADING WITH CACHING
# ============================================================
# The CLARITY dataset is hosted on HuggingFace (ailsntua/QEvasion).
# It contains political press conference Q&A pairs with labels:
#   - 3,448 training examples (with clarity_label)
#   - 308 test examples (no labels — we predict these)
#
# We implement a caching strategy:
#   First run: download from HuggingFace, save to CSV
#   Subsequent runs: load from CSV (no internet needed)
#
# The relevant columns in the dataset are:
#   clarity_label:      the target (Clear Reply / Ambivalent /
#                       Clear Non-Reply)
#   interview_question: the journalist's question
#   interview_answer:   the politician's answer
# There are 17 other columns (metadata) we ignore.

from datasets import load_dataset

CACHE_TRAIN = "/content/train_cached.csv"
CACHE_TEST  = "/content/test_cached.csv"

if os.path.exists(CACHE_TRAIN):
    # Fast path: data already downloaded, load from disk
    train_df = pd.read_csv(CACHE_TRAIN)
    test_df  = pd.read_csv(CACHE_TEST)
    print("Loaded from cache")
else:
    # Slow path: download from HuggingFace and cache
    dataset  = load_dataset("ailsntua/QEvasion")
    train_df = pd.DataFrame(dataset["train"])
    test_df  = pd.DataFrame(dataset["test"])
    train_df.to_csv(CACHE_TRAIN, index=False)
    test_df.to_csv(CACHE_TEST, index=False)
    print("Downloaded and cached")

print("Train:", train_df.shape, "| Test:", test_df.shape)

In [ ]:
# ============================================================
# CELL 3 — COLUMN VERIFICATION
# ============================================================
# Before setting up the full pipeline, we verify the exact
# column names in the dataset and inspect a sample example.
# This cell was added during initial exploration to confirm
# that the column names matched what we expected.
# It also gives a quick sanity check that the data loaded correctly.

LABEL_COL    = "clarity_label"
QUESTION_COL = "interview_question"
ANSWER_COL   = "interview_answer"

print("Label column    :", LABEL_COL)
print("Question column :", QUESTION_COL)
print("Answer column   :", ANSWER_COL)
print()
print("Sample label values:", train_df[LABEL_COL].unique())
print("Sample question:", train_df[QUESTION_COL].iloc[0])
print("Sample answer  :", train_df[ANSWER_COL].iloc[0])

In [ ]:
# ============================================================
# CELL 4 — HYPERPARAMETER CONFIGURATION
# ============================================================
# All hyperparameters are defined in one place for clarity.
#
# MODEL_NAME = "bert-base-uncased":
#   110M parameter transformer, 12 layers, 12 attention heads,
#   hidden size 768. Pretrained on BooksCorpus + Wikipedia.
#   "uncased" means text is lowercased before tokenization.
#
# MAX_LENGTH = 512:
#   BERT's maximum supported input length is 512 tokens.
#   ~38% of our pairs exceed 512 tokens (based on analysis),
#   so the tokenizer truncates those from the end of the answer.
#   We use 512 (not 256) to retain as much text as possible.
#
# BATCH_SIZE = 16:
#   How many examples are processed together in one forward pass.
#   BERT at 512 tokens fits batch=16 on a 16GB T4 GPU.
#
# GRAD_ACCUM = 1:
#   No gradient accumulation needed for BERT (it fits in memory).
#   Effective batch size = 16 * 1 = 16.
#
# WEIGHT_DECAY = 0.01:
#   L2 regularization applied to model weights (but NOT to
#   bias terms and LayerNorm weights — see Cell 6).
#
# PATIENCE = 3:
#   Early stopping: if validation Macro F1 does not improve
#   for 3 consecutive epochs, we stop and restore best weights.

MODEL_NAME   = "bert-base-uncased"
LABEL_COL    = "clarity_label"
QUESTION_COL = "interview_question"
ANSWER_COL   = "interview_answer"
MAX_LENGTH   = 512
BATCH_SIZE   = 16
GRAD_ACCUM   = 1      
NUM_EPOCHS   = 8
LEARNING_RATE = 2e-5
WEIGHT_DECAY  = 0.01
PATIENCE      = 3

print("Model:", MODEL_NAME)

In [ ]:
# ============================================================
# CELL 5 — LABEL ENCODING, TEXT PREP, SPLIT, CLASS WEIGHTS
# ============================================================
# Step 1: Encode string labels as integers.
#   PyTorch's loss functions require integer targets, not strings.
#   label2id: "Ambivalent" -> 0, "Clear Non-Reply" -> 1, etc.
#   id2label: reverse mapping for decoding predictions later.
#   We sort alphabetically so the mapping is consistent across runs.
#
# Step 2: Create text_a and text_b columns.
#   text_a = question, text_b = answer.
#   We pass these as separate arguments to the tokenizer which
#   builds: [CLS] question [SEP] answer [SEP]
#   This is the correct BERT sentence-pair format. Passing them
#   separately (not concatenated) ensures correct token_type_ids:
#     0 for question tokens, 1 for answer tokens.
#
# Step 3: Stratified 80/20 train/val split.
#   stratify=train_df["label_id"] ensures both splits have the
#   same class proportions as the full dataset. This matters
#   because our dataset is severely imbalanced:
#     Ambivalent: 59.2%, Clear Reply: 30.5%, Clear Non-Reply: 10.3%
#   Without stratification, the val set might have very few
#   Clear Non-Reply examples, making evaluation unreliable.
#
# Step 4: Compute class weights for the loss function.
#   compute_class_weight("balanced") calculates:
#     weight_i = N / (n_classes * count_i)
#   where N = total samples and count_i = samples of class i.
#   Result: Ambivalent=0.56, Clear Non-Reply=3.23, Clear Reply=1.09

KNOWN_LABELS = ["Ambivalent", "Clear Non-Reply", "Clear Reply"]
label2id = {l: i for i, l in enumerate(KNOWN_LABELS)}
id2label = {i: l for l, i in label2id.items()}
NUM_LABELS = 3

# Map string labels to integer IDs
train_df["label_id"] = train_df[LABEL_COL].map(label2id)

# Extract and clean text columns
train_df["text_a"] = train_df[QUESTION_COL].astype(str).str.strip()
train_df["text_b"] = train_df[ANSWER_COL].astype(str).str.strip()
test_df["text_a"]  = test_df[QUESTION_COL].astype(str).str.strip()
test_df["text_b"]  = test_df[ANSWER_COL].astype(str).str.strip()

# Stratified 80/20 split — same class proportions in both splits
train_split, val_split = train_test_split(
    train_df, test_size=0.20, random_state=SEED,
    stratify=train_df["label_id"]
)
train_split = train_split.reset_index(drop=True)
val_split   = val_split.reset_index(drop=True)

# Compute class weights from training data only
class_weights = compute_class_weight(
    "balanced",
    classes=np.arange(NUM_LABELS),
    y=train_split["label_id"].values
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)

print(f"Train: {len(train_split)} | Val: {len(val_split)} | Test: {len(test_df)}")
print("Class weights:", class_weights_tensor.tolist())

In [ ]:
# ============================================================
# CELL 6 — PYTORCH DATASET AND DATALOADERS
# ============================================================
# ClarityDataset is a custom PyTorch Dataset class.
# PyTorch requires two methods:
#   __len__: returns total number of samples
#   __getitem__: returns one sample given its index
#
# Inside __getitem__, we tokenize the question-answer pair:
#   - Passing text_a and text_b as two separate arguments
#     tells BERT to build [CLS] question [SEP] answer [SEP]
#     with proper token_type_ids (0=question, 1=answer).
#   - padding="max_length": all sequences padded to MAX_LENGTH
#     so they can be stacked into a rectangular batch tensor.
#   - truncation=True: sequences longer than MAX_LENGTH are cut
#     from the end (the answer tail is truncated).
#   - return_tensors="pt": returns PyTorch tensors directly.
#   - squeeze(0): removes the extra batch dimension that
#     return_tensors="pt" adds. Shape [1, 512] -> [512].
#
# token_type_ids: BERT uses these to distinguish question tokens
#   (type 0) from answer tokens (type 1). We check if they exist
#   before adding them — this way the same class works for
#   DistilBERT (no type IDs) and DeBERTa (no type IDs) too.
#
# is_test=True: test set has no labels, so we skip "labels" key.
#
# DataLoader wraps Dataset and handles:
#   - batching: groups samples into size-16 batches
#   - shuffle=True (train only): randomizes sample order each
#     epoch so the model doesn't overfit to presentation order
#   - num_workers=2: 2 parallel CPU processes load data while
#     GPU is computing, avoiding bottlenecks
#   - pin_memory=True: pins data in CPU memory for faster
#     CPU-to-GPU transfer via DMA

class ClarityDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length, is_test=False):
        self.data       = dataframe.reset_index(drop=True)
        self.tokenizer  = tokenizer
        self.max_length = max_length
        self.is_test    = is_test

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        # Tokenize as sentence pair: [CLS] question [SEP] answer [SEP]
        enc = self.tokenizer(
            str(row["text_a"]),   # question
            str(row["text_b"]),   # answer
            max_length=self.max_length,
            padding="max_length", # pad to fixed length for batching
            truncation=True,      # cut sequences longer than max_length
            return_tensors="pt"   # return PyTorch tensors
        )

        # Build the per-sample dictionary
        item = {
            "input_ids":      enc["input_ids"].squeeze(0),      # [512]
            "attention_mask": enc["attention_mask"].squeeze(0), # [512]
        }

        # BERT produces token_type_ids; DistilBERT/DeBERTa do not
        if "token_type_ids" in enc:
            item["token_type_ids"] = enc["token_type_ids"].squeeze(0)

        # Labels only for train/val, not test
        if not self.is_test:
            item["labels"] = torch.tensor(row["label_id"], dtype=torch.long)

        return item


# Load BERT tokenizer — use_fast=True uses Rust-based fast tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

# Create datasets for each split
train_dataset = ClarityDataset(train_split, tokenizer, MAX_LENGTH, is_test=False)
val_dataset   = ClarityDataset(val_split,   tokenizer, MAX_LENGTH, is_test=False)
test_dataset  = ClarityDataset(test_df,     tokenizer, MAX_LENGTH, is_test=True)

# Wrap in DataLoaders for batched iteration
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)

print("Train batches:", len(train_loader),
      "| Val:", len(val_loader),
      "| Test:", len(test_loader))

In [ ]:
# ============================================================
# CELL 7 — MODEL, LOSS, OPTIMIZER, SCHEDULER
# ============================================================
# Step 1: Free GPU memory from any previous runs.
#   gc.collect() triggers Python's garbage collector.
#   torch.cuda.empty_cache() releases the PyTorch memory cache.
#   This prevents OOM errors when running multiple experiments.
#
# Step 2: Load BERT with a 3-class classification head.
#   AutoModelForSequenceClassification loads the pretrained
#   BERT transformer and replaces its MLM head with a new
#   linear layer: hidden_size (768) -> num_labels (3).
#
# Step 3: Weighted cross-entropy loss.
#   nn.CrossEntropyLoss(weight=...) multiplies each sample's
#   loss by the weight of its true class before averaging.
#
# Step 4: AdamW optimizer with selective weight decay.
#   We group parameters into two sets:
#   - Parameters WITH weight decay: all transformer weights
#     (attention matrices, FFN weights, embedding weights)
#   - Parameters WITHOUT weight decay: bias terms and LayerNorm
#     weights. These don't benefit from L2 regularization and
#     adding it can slightly hurt performance.
#   This is the standard setup from the original BERT paper.
#
# Step 5: Linear warmup + linear decay LR schedule.
#   total_steps = batches_per_epoch * num_epochs
#   For BERT: no gradient accumulation, so total_steps =
#   len(train_loader) * NUM_EPOCHS directly.
#   warmup_steps = 10% of total steps.
#   During warmup: LR linearly goes from 0 -> LEARNING_RATE.
#   After warmup: LR linearly goes from LEARNING_RATE -> 0.

import gc
gc.collect()
torch.cuda.empty_cache()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load BERT pretrained weights + new 3-class classification head
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,    # stored in model config for convenience
    label2id=label2id,
).to(DEVICE)

# Weighted cross-entropy: rare classes penalized more heavily
loss_fn = torch.nn.CrossEntropyLoss(
    weight=class_weights_tensor.to(DEVICE)
)

# AdamW: weight decay on most params, NOT on bias/LayerNorm
no_decay = ["bias", "LayerNorm.weight"]
optimizer = torch.optim.AdamW([
    {"params": [p for n,p in model.named_parameters()
                if not any(nd in n for nd in no_decay)],
     "weight_decay": WEIGHT_DECAY},
    {"params": [p for n,p in model.named_parameters()
                if     any(nd in n for nd in no_decay)],
     "weight_decay": 0.0},
], lr=LEARNING_RATE)

# Linear warmup + decay scheduler
total_steps  = len(train_loader) * NUM_EPOCHS
warmup_steps = int(0.1 * total_steps)  # 10% of training as warmup
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

print("Device:", DEVICE)
print(f"Params: {sum(p.numel() for p in model.parameters()):,}")
print(f"Total steps: {total_steps} | Warmup: {warmup_steps}")

In [ ]:
# ============================================================
# CELL 8 — EVALUATION FUNCTION
# ============================================================
# This reusable function evaluates the model on any DataLoader
# (validation or test) without updating weights.
#
# model.eval():
#   Switches the model to evaluation mode. This disables
#   dropout layers (which randomly zero activations during
#   training) and sets BatchNorm to use running statistics.
#   Without this, dropout would give noisy/inconsistent
#   evaluation results each time.
#
# torch.no_grad():
#   Disables gradient computation and tracking entirely.
#   During evaluation we never call backward(), so we don't
#   need gradients. This saves significant memory and time.
#
# token_type_ids check:
#   BERT produces token_type_ids (segment IDs: 0=question,
#   1=answer). We pass them if they exist. This way the same
#   evaluate() function works for BERT, DistilBERT, and DeBERTa.
#
# out.logits:
#   Raw unnormalized scores for each class. Shape: (batch, 3).
#   argmax picks the class with the highest score.
#   We don't need softmax/probabilities for prediction.
#
# Metrics explanation:
#   accuracy:   fraction of correctly classified examples
#   F1 weighted: F1 score weighted by class frequency (support)
#   Macro F1:   unweighted mean of per-class F1 scores
#               This is the PRIMARY metric for this competition
#               because it treats all 3 classes equally,
#               regardless of how common or rare they are.
#   precision:  of all predicted positives, how many are correct
#   recall:     of all actual positives, how many were found
#
# all_preds, all_labels are returned for confusion matrix
# and error analysis in the final cell.

def evaluate(model, loader, loss_fn, device):
    model.eval()  # disable dropout for consistent results
    total_loss, all_preds, all_labels = 0, [], []

    with torch.no_grad():  # no gradient tracking needed
        for batch in loader:
            ids  = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            lbls = batch["labels"].to(device)

            # Build kwargs: include token_type_ids only if present
            kwargs = {"input_ids": ids, "attention_mask": mask}
            if "token_type_ids" in batch:
                kwargs["token_type_ids"] = batch["token_type_ids"].to(device)

            out = model(**kwargs)          # forward pass
            # out.logits shape: (batch_size, 3)

            total_loss += loss_fn(out.logits, lbls).item()

            # argmax -> predicted class index for each sample
            all_preds.extend(
                torch.argmax(out.logits, dim=1).cpu().numpy()
            )
            all_labels.extend(lbls.cpu().numpy())

    n = len(loader)
    return (
        total_loss / n,                                                    # avg loss
        accuracy_score(all_labels, all_preds),                             # accuracy
        f1_score(all_labels, all_preds, average="weighted"),               # weighted F1
        f1_score(all_labels, all_preds, average="macro"),                  # MACRO F1 (primary)
        precision_score(all_labels, all_preds, average="weighted",
                        zero_division=0),                                  # precision
        recall_score(all_labels, all_preds, average="weighted",
                     zero_division=0),                                     # recall
        all_preds,                                                         # raw predictions
        all_labels                                                         # raw true labels
    )

print("evaluate() ready")

In [ ]:
# ============================================================
# CELL 9 — CUSTOM TRAINING LOOP
# ============================================================
# We implement the training loop from scratch as required
# by the assignment 
#
# EARLY STOPPING:
#   We monitor validation Macro F1 after each epoch.
#   If it doesn't improve for PATIENCE=3 consecutive epochs,
#   we stop and restore the best weights seen so far.
#   best_model_state stores a DEEP COPY of the model weights
#   at the best epoch: {k: v.clone()} is critical because
#   PyTorch tensors are references — without clone(), the dict
#   would just hold pointers that change as training continues.
#
# TRAINING LOOP STRUCTURE (for BERT, GRAD_ACCUM=1):
#   For each epoch:
#     model.train() — re-enable dropout
#     optimizer.zero_grad() — clear old gradients at epoch start
#     For each batch:
#       1. Move data to GPU
#       2. Forward pass: compute logits = model(input)
#       3. Compute weighted cross-entropy loss
#       4. loss.backward() — compute gradients
#       5. clip_grad_norm_() — prevent gradient explosion
#       6. optimizer.step() — update weights
#       7. scheduler.step() — update learning rate
#       8. optimizer.zero_grad() — clear gradients for next step
#     Evaluate on validation set
#     Check early stopping
#
# NOTE on optimizer.zero_grad() placement:
#   For BERT (GRAD_ACCUM=1), we zero gradients INSIDE the batch
#   loop after each update. This differs from DeBERTa where
#   we accumulate for 4 steps before updating.
#
# GRADIENT CLIPPING:
#   clip_grad_norm_(model.parameters(), 1.0) caps the L2 norm
#   of all gradients at 1.0. This prevents "exploding gradients"
#   where very large gradients from certain layers would cause
#   the model weights to jump wildly and destabilize training.
#   It's applied AFTER backward() but BEFORE optimizer.step().
#
# scheduler.get_last_lr()[0] prints the current learning rate,
# showing the warmup increase and later decay clearly.

best_macro_f1, best_model_state, patience_counter = 0.0, None, 0
history = {"train_loss": [], "val_loss": [], "val_f1": [],
           "val_macro_f1": [], "val_accuracy": []}

print(f"Training {MODEL_NAME} for {NUM_EPOCHS} epochs (LR={LEARNING_RATE})...")
print("="*65)

for epoch in range(NUM_EPOCHS):
    t0 = time.time()
    model.train()           # re-enable dropout
    total_train_loss = 0
    optimizer.zero_grad()   # clear any leftover gradients

    for step, batch in enumerate(train_loader):
        # Move all tensors to GPU
        ids  = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        lbls = batch["labels"].to(DEVICE)
        kwargs = {"input_ids": ids, "attention_mask": mask}
        if "token_type_ids" in batch:
            kwargs["token_type_ids"] = batch["token_type_ids"].to(DEVICE)

        # Forward pass: get raw class scores
        logits = model(**kwargs).logits  # shape: (batch_size, 3)

        # Compute weighted cross-entropy loss
        loss = loss_fn(logits, lbls)
        loss.backward()            # compute gradients
        total_train_loss += loss.item()

        # Clip gradients to prevent explosion (norm capped at 1.0)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()    # update model weights with gradients
        scheduler.step()    # update learning rate
        optimizer.zero_grad()  # clear gradients for next batch

        # Print progress every 50 steps
        if (step + 1) % 50 == 0:
            print(f"  Epoch {epoch+1} | Step {step+1}/{len(train_loader)}"
                  f" | Loss: {loss.item():.4f}")

    # End of epoch: run validation
    avg_train_loss = total_train_loss / len(train_loader)
    val_loss, val_acc, val_f1, val_macro_f1, _, _, _, _ = \
        evaluate(model, val_loader, loss_fn, DEVICE)

    # Record metrics for plotting
    history["train_loss"].append(avg_train_loss)
    history["val_loss"].append(val_loss)
    history["val_f1"].append(val_f1)
    history["val_macro_f1"].append(val_macro_f1)
    history["val_accuracy"].append(val_acc)

    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS} — {time.time()-t0:.0f}s")
    print(f"  Train Loss: {avg_train_loss:.4f} | Val Loss: {val_loss:.4f}")
    print(f"  Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}"
          f" | Val Macro F1: {val_macro_f1:.4f}")

    # Early stopping: track best Macro F1
    if val_macro_f1 > best_macro_f1:
        best_macro_f1 = val_macro_f1
        # Deep copy weights (clone() required — tensors are references)
        best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
        patience_counter = 0
        print(f"  Best Macro F1: {best_macro_f1:.4f} — weights saved")
    else:
        patience_counter += 1
        print(f"  No improvement. Patience: {patience_counter}/{PATIENCE}")
        if patience_counter >= PATIENCE:
            print("Early stopping triggered.")
            break
    print("-"*65)

# Restore the best weights found during training
model.load_state_dict(best_model_state)
print(f"\nTraining complete. Best Val Macro F1: {best_macro_f1:.4f}")

In [ ]:
# ============================================================
# CELL 10 — PLOTS, FINAL EVALUATION, AND SUBMISSION
# ============================================================
# Part A: Training curves 
#
#   Plot 1 — Loss curve:
#     Shows train loss and val loss per epoch.
#     How to read it:
#       Both decreasing together = healthy training
#       Val loss rising while train loss falls = OVERFITTING
#         (model memorizing training data, not generalizing)
#       Both flat and high = UNDERFITTING
#         (model not learning, needs more capacity or longer training)
#
#   Plot 2 — Metrics curve:
#     Shows val F1 (weighted), val Macro F1, val Accuracy per epoch.
#     Macro F1 is the primary metric — we monitor this for
#     early stopping and for comparing models.
#
# Part B: Final evaluation
#   Runs evaluate() with the best model weights on the validation
#   set and prints a full per-class classification report:
#   precision, recall, F1, support for each of the 3 classes.
#
# Part C: Test set prediction
#   The test set has 308 examples with no ground truth labels.
#   We run the trained model and collect predictions.
#   model.eval() + torch.no_grad() for efficient inference.
#   torch.argmax(logits, dim=1): picks the class with highest
#   raw score for each example in the batch.
#   id2label converts integer predictions to string labels.
#
# Part D: Submission CSV
#   Required format:
#     Id: integer 0, 1, 2, ..., 307
#     Predicted: string label
#   Filename must match exactly: submission_bert-base-uncased.csv

# --- Part A: Training curves ---
epochs_ran = len(history["train_loss"])
x = range(1, epochs_ran + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(x, history["train_loss"], "o-", label="Train loss")
axes[0].plot(x, history["val_loss"],   "o-", label="Val loss")
axes[0].set_title(f"{MODEL_NAME} — Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(x, history["val_f1"],       "o-", label="Val F1 (weighted)")
axes[1].plot(x, history["val_macro_f1"], "o-", label="Val Macro F1")
axes[1].plot(x, history["val_accuracy"], "o-", label="Val Accuracy")
axes[1].set_title(f"{MODEL_NAME} — Metrics")
axes[1].set_xlabel("Epoch")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("/kaggle/working/training_curves_bert.png",
            dpi=150, bbox_inches="tight")
plt.show()

# --- Part B: Final evaluation on validation set ---
val_loss, val_acc, val_f1, val_macro_f1, val_prec, val_rec, val_preds, val_true = \
    evaluate(model, val_loader, loss_fn, DEVICE)

print("="*55)
print(f"  Accuracy : {val_acc:.4f}")
print(f"  Precision: {val_prec:.4f}  (weighted)")
print(f"  Recall   : {val_rec:.4f}  (weighted)")
print(f"  F1       : {val_f1:.4f}  (weighted)")
print(f"  Macro F1 : {val_macro_f1:.4f}")
print("="*55)
# Per-class breakdown: how well did we do on each of the 3 classes?
print(classification_report(
    val_true, val_preds,
    target_names=[id2label[i] for i in range(NUM_LABELS)]
))

# --- Part C: Generate test set predictions ---
model.eval()  # disable dropout
all_test_preds = []

with torch.no_grad():  # no gradients needed for inference
    for batch in test_loader:
        ids  = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        kwargs = {"input_ids": ids, "attention_mask": mask}
        if "token_type_ids" in batch:
            kwargs["token_type_ids"] = batch["token_type_ids"].to(DEVICE)
        # argmax picks highest logit -> predicted class index
        all_test_preds.extend(
            torch.argmax(model(**kwargs).logits, dim=1).cpu().numpy()
        )

# --- Part D: Build and save submission CSV ---
submission = pd.DataFrame({
    "Id":        range(len(all_test_preds)),            # 0 to 307
    "Predicted": [id2label[p] for p in all_test_preds]  # string labels
})
submission.to_csv("/kaggle/working/submission_bert-base-uncased.csv",
                  index=False)  # index=False: don't write row numbers

print("Submission saved!")
print(submission["Predicted"].value_counts())